In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv")

df[df["genres"].isna()].shape

(160910, 7)

In [5]:
df[df["genres"].isna()]["book_id"].nunique()

585

In [6]:
import pandas as pd
import ast

print("\nDataset shape:")
print(df.shape)


# -------------------------------------------------
# Genre parser (same logic as your SVD script)
# -------------------------------------------------
def parse_genres(s):

    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip().strip('"').strip("'") for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip().strip('"').strip("'") for t in s.split(sep) if t.strip()]

    return [s.strip().strip('"').strip("'")]


# -------------------------------------------------
# Collect statistics
# -------------------------------------------------

all_genres = set()
genre_book_map = {}
books_with_no_genre = set()

for _, row in df.iterrows():

    book = row["book_id"]
    parsed = parse_genres(row["genres"])

    if len(parsed) == 0:
        books_with_no_genre.add(book)
        continue

    for g in parsed:

        all_genres.add(g)

        if g not in genre_book_map:
            genre_book_map[g] = set()

        genre_book_map[g].add(book)


# -------------------------------------------------
# Unique genres
# -------------------------------------------------

print("\nUnique genres:")
for g in sorted(all_genres):
    print(g)

print("\nNumber of unique genres:", len(all_genres))


# -------------------------------------------------
# Unique books per genre
# -------------------------------------------------

print("\nUnique books per genre:")

genre_counts = []

for g in sorted(genre_book_map.keys()):
    count = len(genre_book_map[g])
    genre_counts.append((g, count))

genre_df = pd.DataFrame(genre_counts, columns=["genre", "unique_books"])
genre_df = genre_df.sort_values("unique_books", ascending=False)

print(genre_df)


# -------------------------------------------------
# Missing genres
# -------------------------------------------------

print("\nBooks with missing / unparsable genres:")
print(len(books_with_no_genre))


# -------------------------------------------------
# Dataset summary
# -------------------------------------------------

print("\nTotal unique books in dataset:")
print(df["book_id"].nunique())


Dataset shape:
(5976479, 7)

Unique genres:
Adult
Adventure
Children's
Classics
Drama
Fantasy
Historical
Horror
Mystery
Nonfiction
Romance
Science Fiction
Thriller

Number of unique genres: 13

Unique books per genre:
              genre  unique_books
4             Drama          3006
8           Mystery          2563
10          Romance          2131
5           Fantasy          2088
1         Adventure          1789
12         Thriller          1606
9        Nonfiction          1071
3          Classics           901
2        Children's           863
6        Historical           857
11  Science Fiction           855
7            Horror           769
0             Adult           331

Books with missing / unparsable genres:
585

Total unique books in dataset:
10000


In [8]:
import pandas as pd
import ast



def parse_genres(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# rows where genre parsing fails
invalid_rows = df[df["genres"].apply(lambda x: len(parse_genres(x)) == 0)]

print("Number of rows with invalid genres:", len(invalid_rows))

print("\nExample rows (full rows):\n")
print(invalid_rows.head(10))

Number of rows with invalid genres: 160910

Example rows (full rows):

      user_id  book_id  rating         decade original_title authors genres
1           2     4081       4           2000            NaN     NaN    NaN
433        32     1383       3           1970            NaN     NaN    NaN
438        32       75       3           1990            NaN     NaN    NaN
584        31       75       3           1990            NaN     NaN    NaN
1003       76       75       5           1990            NaN     NaN    NaN
1034       25     8555       4           1960            NaN     NaN    NaN
1297       78     4106       4  Ancient Books            NaN     NaN    NaN
1561       24     2757       3           1990            NaN     NaN    NaN
1619      113     3244       5           2000            NaN     NaN    NaN
1637      113     5629       4           1990            NaN     NaN    NaN


## clean dataset

In [9]:
import pandas as pd
import ast

INPUT_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv"
OUTPUT_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv"

df = pd.read_csv(INPUT_PATH)

print("Original dataset shape:", df.shape)


# -------------------------------------------------
# Genre parser (same logic as your SVD code)
# -------------------------------------------------
def parse_genres(s):

    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# -------------------------------------------------
# Identify books with no valid genre
# -------------------------------------------------

df["parsed_genres"] = df["genres"].apply(parse_genres)

invalid_books = df[df["parsed_genres"].apply(len) == 0]["book_id"].unique()

print("Books with no valid genre:", len(invalid_books))


# -------------------------------------------------
# Remove those books from the dataset
# -------------------------------------------------

df_clean = df[~df["book_id"].isin(invalid_books)].copy()

# drop helper column
df_clean.drop(columns=["parsed_genres"], inplace=True)

print("Clean dataset shape:", df_clean.shape)


# -------------------------------------------------
# Save cleaned dataset
# -------------------------------------------------

df_clean.to_csv(OUTPUT_PATH, index=False)

print("\nClean dataset saved as:", OUTPUT_PATH)

Original dataset shape: (5976479, 7)
Books with no valid genre: 585
Clean dataset shape: (5815569, 7)

Clean dataset saved as: /home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv


In [10]:
import pandas as pd
import ast

df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv")

def parse_genres(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# rows where genre parsing fails
invalid_rows = df[df["genres"].apply(lambda x: len(parse_genres(x)) == 0)]

print("Number of rows with invalid genres:", len(invalid_rows))

print("\nExample rows (full rows):\n")
print(invalid_rows.head(10))

Number of rows with invalid genres: 0

Example rows (full rows):

Empty DataFrame
Columns: [user_id, book_id, rating, decade, original_title, authors, genres]
Index: []


In [6]:
df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv")

df[df["genres"].isna()].shape

(0, 7)

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

# =========================
# Paths
# =========================

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)



# =========================
# Split genre pairs
# Assumes each book has exactly two genres like:
# "Mystery, Historical"
# =========================
genres_split = df["genres"].astype(str).str.split(",", n=1, expand=True)
df["g1"] = genres_split[0].str.strip()
df["g2"] = genres_split[1].str.strip() if 1 in genres_split.columns else ""

# Keep one row per book for domain analysis
books = df[["book_id", "g1", "g2"]].drop_duplicates(subset=["book_id"]).copy()

total_books = books["book_id"].nunique()

# =========================
# Counts
# =========================
first_counts = books["g1"].value_counts()
second_counts = books["g2"].value_counts()

all_genres = sorted(set(first_counts.index).union(set(second_counts.index)))

first_counts = first_counts.reindex(all_genres, fill_value=0)
second_counts = second_counts.reindex(all_genres, fill_value=0)
total_counts = first_counts + second_counts

# percentage of all genre appearances
# each book contributes 2 genre appearances
grand_total_genre_appearances = total_counts.sum()
pct_all = (total_counts / grand_total_genre_appearances) * 100

# percentage of that genre that appears in first position
pct_first_of_genre = (first_counts / total_counts * 100).fillna(0)
pct_second_of_genre = (second_counts / total_counts * 100).fillna(0)

# =========================
# Summary table
# =========================
summary = pd.DataFrame({
    "genre": all_genres,
    "first": first_counts.values,
    "second": second_counts.values,
    "total": total_counts.values,
    "percentage_of_all": pct_all.values,
    "percentage_of_that_genre_first": pct_first_of_genre.values,
    "percentage_of_that_genre_second": pct_second_of_genre.values,
}).sort_values("total", ascending=False)

summary.to_csv(OUT_DIR / "genre_summary_table.csv", index=False)

# =========================
# Mild, distinct colors
# =========================
genre_color_map = {
    "Adult": "#8ECae6",
    "Adventure": "#FFB703",
    "Children's": "#B8DE6F",
    "Classics": "#CDB4DB",
    "Drama": "#F4A261",
    "Fantasy": "#90BE6D",
    "Historical": "#E5989B",
    "Horror": "#6D597A",
    "Mystery": "#84A59D",
    "Nonfiction": "#A8DADC",
    "Romance": "#F28482",
    "Science Fiction": "#577590",
    "Thriller": "#F6BD60",
}

bar_colors = [genre_color_map[g] for g in summary["genre"]]

# =========================
# Helper: add labels on bars
# =========================
def add_bar_labels(ax, fmt="{:,.0f}", fontsize=9, rotation=0, pad=3):
    for container in ax.containers:
        labels = []
        for bar in container:
            h = bar.get_height()
            if pd.isna(h) or h <= 0:
                labels.append("")
            else:
                labels.append(fmt.format(h))
        ax.bar_label(container, labels=labels, padding=pad, fontsize=fontsize, rotation=rotation)

# =========================
# Figure 1: Total books per genre (bar)
# =========================
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.bar(summary["genre"], summary["total"], color=bar_colors, edgecolor="black", linewidth=0.7)

ax.set_title("Total Number of Unique Books per Genre", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Unique Books")
ax.tick_params(axis="x", rotation=45)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + max(summary["total"]) * 0.01,
        f"{int(h)}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_total_books_per_genre_bar.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Figure 2: Genre percentage of all (pie)
# =========================
fig, ax = plt.subplots(figsize=(10, 10))
ax.pie(
    summary["total"],
    labels=summary["genre"],
    autopct="%1.1f%%",
    startangle=90,
    colors=bar_colors,
    wedgeprops={"edgecolor": "white", "linewidth": 1}
)
ax.set_title("Percentage of All Genre Appearances", fontsize=14)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig2_genre_percentage_pie.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Figure 3: First vs second genre counts (grouped bar)
# =========================
plot_df = summary.copy()
x = range(len(plot_df))
width = 0.38

first_color = "#7FB3D5"
second_color = "#F5B7B1"

fig, ax = plt.subplots(figsize=(13, 7))
bars1 = ax.bar([i - width/2 for i in x], plot_df["first"], width=width, label="First genre",
               color=first_color, edgecolor="black", linewidth=0.7)
bars2 = ax.bar([i + width/2 for i in x], plot_df["second"], width=width, label="Second genre",
               color=second_color, edgecolor="black", linewidth=0.7)

ax.set_title("Genre Counts by Position", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Unique Books")
ax.set_xticks(list(x))
ax.set_xticklabels(plot_df["genre"], rotation=45, ha="right")
ax.legend()

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + max(plot_df["total"]) * 0.008, f"{int(h)}",
            ha="center", va="bottom", fontsize=8)

for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + max(plot_df["total"]) * 0.008, f"{int(h)}",
            ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig3_first_vs_second_grouped_bar.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Figure 4: Stacked bar (first + second)
# =========================
fig, ax = plt.subplots(figsize=(13, 7))

bars_first = ax.bar(plot_df["genre"], plot_df["first"], color=first_color, edgecolor="black",
                    linewidth=0.7, label="First genre")
bars_second = ax.bar(plot_df["genre"], plot_df["second"], bottom=plot_df["first"],
                     color=second_color, edgecolor="black", linewidth=0.7, label="Second genre")

ax.set_title("First and Second Genre Distribution per Genre", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Unique Books")
ax.tick_params(axis="x", rotation=45)
ax.legend()

# total on top
for i, total in enumerate(plot_df["total"]):
    ax.text(i, total + max(plot_df["total"]) * 0.01, f"{int(total)}",
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig4_first_second_stacked_bar.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Figure 5: Table as image
# =========================
table_df = summary.copy()
table_df["percentage_of_all"] = table_df["percentage_of_all"].round(2)
table_df["percentage_of_that_genre_first"] = table_df["percentage_of_that_genre_first"].round(2)
table_df["percentage_of_that_genre_second"] = table_df["percentage_of_that_genre_second"].round(2)

fig, ax = plt.subplots(figsize=(16, 7))
ax.axis("off")

tbl = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    loc="center",
    cellLoc="center"
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.15, 1.45)

ax.set_title("Genre Summary Table", fontsize=14, pad=20)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig5_genre_summary_table.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Figure 6: Violin plot
# Distribution of genre position encoded as:
# 1 = first genre, 2 = second genre
# This shows where each genre tends to appear
# =========================
violin_data = []
violin_labels = []

for g in summary["genre"]:
    vals = [1] * int(first_counts[g]) + [2] * int(second_counts[g])
    if len(vals) > 0:
        violin_data.append(vals)
        violin_labels.append(g)

fig, ax = plt.subplots(figsize=(13, 7))
parts = ax.violinplot(violin_data, showmeans=True, showmedians=True, showextrema=True)

# color each violin differently
for i, body in enumerate(parts["bodies"]):
    genre = violin_labels[i]
    body.set_facecolor(genre_color_map[genre])
    body.set_edgecolor("black")
    body.set_alpha(0.8)

for partname in ("cbars", "cmins", "cmaxes", "cmeans", "cmedians"):
    if partname in parts:
        parts[partname].set_edgecolor("black")
        parts[partname].set_linewidth(1)

ax.set_xticks(range(1, len(violin_labels) + 1))
ax.set_xticklabels(violin_labels, rotation=45, ha="right")
ax.set_yticks([1, 2])
ax.set_yticklabels(["First", "Second"])
ax.set_ylabel("Genre Position")
ax.set_xlabel("Genre")
ax.set_title("Violin Plot of Genre Position (First vs Second)", fontsize=14)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig6_genre_position_violin.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Optional: save a cleaner CSV table with requested columns only
# genre, first, second, total, percentage of all, percentage of that genre
# here "percentage of that genre" = percentage of that genre appearing first
# =========================
requested_table = summary[[
    "genre", "first", "second", "total", "percentage_of_all", "percentage_of_that_genre_first"
]].copy()

requested_table = requested_table.rename(columns={
    "percentage_of_all": "percentage_of_all",
    "percentage_of_that_genre_first": "percentage_of_that_genre"
})

requested_table.to_csv(OUT_DIR / "genre_requested_table.csv", index=False)

print("Saved files to:")
print(OUT_DIR)
print("\nFiles created:")
print("- fig1_total_books_per_genre_bar.png")
print("- fig2_genre_percentage_pie.png")
print("- fig3_first_vs_second_grouped_bar.png")
print("- fig4_first_second_stacked_bar.png")
print("- fig5_genre_summary_table.png")
print("- fig6_genre_position_violin.png")
print("- genre_summary_table.csv")
print("- genre_requested_table.csv")

Saved files to:
/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures

Files created:
- fig1_total_books_per_genre_bar.png
- fig2_genre_percentage_pie.png
- fig3_first_vs_second_grouped_bar.png
- fig4_first_second_stacked_bar.png
- fig5_genre_summary_table.png
- fig6_genre_position_violin.png
- genre_summary_table.csv
- genre_requested_table.csv
